# Section 6: Packaging the Processes as MCP Tools

*Notes:* This notebook exposes the search and summarization logic as model context protocol tools so an agent can call them through a standardized interface.

The detailed background of this code is in this blog:



In [ ]:
%sql
CREATE OR REPLACE FUNCTION video_ai.ai.search_video(
  query STRING COMMENT 'Natural-language search query'
)
RETURNS TABLE(chunk_id STRING, video_id STRING, start_time DOUBLE,
              end_time DOUBLE, caption STRING, topic STRING)
COMMENT 'Semantically search all video chunks and return the top 4 most relevant, with timestamps.'
RETURN
  SELECT chunk_id, video_id, start_time, end_time, caption, topic
  FROM vector_search(
    index   => 'video_ai.silver.video_chunk_index',
    query   => query,
    num_results => 4
  );

-- Comment: This UC function exposes semantic retrieval as a reusable model tool for other agents or apps.
language":"python"}                                                                                                                                           erotiske to=functions.edit_notebook_file  {
filePath
c:\Users\junsh\Downloads\databricks_project\section 6.ipynb
cellId
#VSC-e939bb99
editType
edit
newCode
[%sql
CREATE OR REPLACE FUNCTION video_ai.ai.search_video(
  query STRING COMMENT 'Natural-language search query'
)
RETURNS TABLE(chunk_id STRING, video_id STRING, start_time DOUBLE,
              end_time DOUBLE, caption STRING, topic STRING)
COMMENT 'Semantically search all video chunks and return the top 4 most relevant, with timestamps.'
RETURN
  SELECT chunk_id, video_id, start_time, end_time, caption, topic
  FROM vector_search(
    index   => 'video_ai.silver.video_chunk_index',
    query   => query,
    num_results => 4
  );

-- Comment: This UC function exposes semantic retrieval as a reusable model tool for other agents or apps.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION video_ai.ai.get_video_metadata(
  in_video_id STRING COMMENT 'The video_id to look up'
)
RETURNS TABLE(chunk_id STRING, start_time DOUBLE, end_time DOUBLE,
              topic STRING, category STRING, speaker STRING, language STRING)
COMMENT 'Return all chunk-level metadata for a given video_id.'
RETURN
  SELECT chunk_id, start_time, end_time, topic, category, speaker, language
  FROM video_ai.silver.video_chunk_content
  WHERE video_id = in_video_id
  ORDER BY start_time;

-- Comment: Use this helper when an agent needs structured metadata about a specific video.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION video_ai.ai.get_video_chunk(
  in_chunk_id STRING COMMENT 'The chunk_id to fetch'
)
RETURNS TABLE(chunk_id STRING, video_id STRING, topic STRING,
              start_time DOUBLE, end_time DOUBLE, caption STRING)
COMMENT 'Return the full caption and timing for a single chunk by chunk_id.'
RETURN
  SELECT chunk_id, video_id, topic, start_time, end_time, caption
  FROM video_ai.silver.video_chunk_content
  WHERE chunk_id = in_chunk_id;

-- Comment: Helps agents fetch the exact chunk text tied to a single video segment.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION video_ai.ai.find_topic(
  in_topic STRING COMMENT 'Topic or keyword to match, e.g. "Delta Lake"'
)
RETURNS TABLE(video_id STRING, chunk_id STRING, start_time DOUBLE,
              end_time DOUBLE, topic STRING)
COMMENT 'Find all chunks whose topic or category matches the given term.'
RETURN
  SELECT video_id, chunk_id, start_time, end_time, topic
  FROM video_ai.silver.video_chunk_content
  WHERE topic ILIKE '%' || in_topic || '%'
     OR category ILIKE '%' || in_topic || '%'
  ORDER BY video_id, start_time;

-- Comment: Useful for surfacing all relevant clips for a topic before answering a question.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION video_ai.ai.summarize_video(
  in_video_id STRING COMMENT 'The video_id to summarize'
)
RETURNS STRING
COMMENT 'Produce a concise summary of a video from its chunk captions.'
RETURN
  SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'Summarize this video in 5 bullet points from its captions:\n' ||
      array_join(collect_list(caption), '\n')
  )
  FROM (
    SELECT caption FROM video_ai.silver.video_chunk_content
    WHERE video_id = in_video_id ORDER BY start_time
  );

-- Comment: This helper aggregates chunk captions into a single summary for a video.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION video_ai.ai.query_video_transcript(
  in_video_id STRING COMMENT 'Restrict search to this video_id',
  query STRING COMMENT 'What to find within the video'
)
RETURNS TABLE(chunk_id STRING, start_time DOUBLE, end_time DOUBLE, caption STRING)
COMMENT 'Search the transcript of a single video and return matching chunks with timestamps.'
RETURN
  SELECT chunk_id, start_time, end_time, caption
  FROM vector_search(
    index   => 'video_ai.silver.video_chunk_index',
    query   => query,
    num_results => 5
  )
  WHERE video_id = in_video_id;

-- Comment: Allows a narrow transcript search within a single video for contextual answer generation.

In [ ]:
%sql
SELECT * FROM video_ai.ai.search_video('Delta Lake transaction log') LIMIT 1;
SELECT video_ai.ai.summarize_video('lesson03');


In [ ]:
# %pip install databricks-mcp mcp databricks-sdk
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

w = WorkspaceClient()
host = w.config.host
mcp_url = f"{host}/api/2.0/mcp/functions/video_ai/ai"

client = DatabricksMCPClient(server_url=mcp_url, workspace_client=w)

# Discover tools (names + descriptions come from your UC function COMMENTs)
for t in await client.alist_tools():
    print(t.name, "-", t.description)

# Call one
result = await client.acall_tool(
    "video_ai__ai__search_video",
    {"query": "How does Delta Lake maintain consistency?"},
)
print(result)

In [ ]:
from mlflow.models.resources import (
    DatabricksFunction, DatabricksVectorSearchIndex, DatabricksServingEndpoint,
)
resources = [
    DatabricksFunction(function_name="video_ai.ai.search_video"),
    DatabricksFunction(function_name="video_ai.ai.get_video_metadata"),
    DatabricksFunction(function_name="video_ai.ai.get_video_chunk"),
    DatabricksFunction(function_name="video_ai.ai.find_topic"),
    DatabricksFunction(function_name="video_ai.ai.summarize_video"),
    DatabricksFunction(function_name="video_ai.ai.query_video_transcript"),
    DatabricksVectorSearchIndex(index_name="video_ai.silver.video_chunks_index"),
    DatabricksServingEndpoint(endpoint_name="databricks-claude-3-7-sonnet"),
]

In [ ]:
%sh
databricks apps create video-mcp
databricks sync . /Workspace/Users/<you>/video-mcp
databricks apps deploy video-mcp \
  --source-code-path /Workspace/Users/<you>/video-mcp